# 07 · LightGBM baseline

Untuned LightGBM on the same v1 windows. Primary candidate family before Optuna.

In [ ]:
import pandas as pd

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import train_lightgbm
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics, log_parameters

nb = setup_model_session()
train = load_split("v1", "train", nb.config, engine=nb.engine)
valid = load_split("v1", "validation", nb.config, engine=nb.engine)
test = load_split("v1", "test", nb.config, engine=nb.engine)

In [ ]:
model = train_lightgbm(
    train,
    target_vector(train),
    valid,
    target_vector(valid),
    threshold=nb.threshold,
)
valid_metrics = quality_metrics(target_vector(valid), model.predict_proba(valid), threshold=nb.threshold)
test_metrics = quality_metrics(target_vector(test), model.predict_proba(test), threshold=nb.threshold)
pd.DataFrame([valid_metrics, test_metrics], index=["validation", "test"])

In [ ]:
path = model.save(nb.artifacts / "models" / "v1_lightgbm.joblib")
with clearml_task("train_lightgbm", config=nb.config, task_type="training", tags=["v1", "lightgbm"], init=True) as task:
    log_parameters(task, model.params)
    log_metrics(task, test_metrics, title="v1_test")
path